<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/05-data-sources-io/00-file-formats-and-options.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 5.0 — File formats and the options that matter

Writes go to a local `output/` folder next to this notebook; the last
cell removes it. Reminder: local writes on Windows need the lesson-1.2
winutils setup.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("5.0-formats")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True).schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)

## A write is a DIRECTORY of part-files

In [ ]:
orders.write.mode("overwrite").parquet("output/orders_parquet")

import pathlib
for p in sorted(pathlib.Path("output/orders_parquet").iterdir()):
    print(f"{p.name:<55} {p.stat().st_size:>7} bytes")
print("\npart-file count == partition count:", orders.rdd.getNumPartitions())

## Parquet round-trips types; CSV does not

In [ ]:
spark.read.parquet("output/orders_parquet").printSchema()   # schema lives IN the file

orders.write.mode("overwrite").option("header", True).csv("output/orders_csv")
spark.read.option("header", True).csv("output/orders_csv").printSchema()  # all strings again

## Size on disk: same 41 rows, three ways

In [ ]:
orders.coalesce(1).write.mode("overwrite").option("header", True).csv("output/sz_csv")
orders.coalesce(1).write.mode("overwrite").parquet("output/sz_parquet")
orders.coalesce(1).write.mode("overwrite").option("compression", "zstd").parquet("output/sz_zstd")

def dir_size(d):
    return sum(p.stat().st_size for p in pathlib.Path(d).rglob("*") if p.is_file() and not p.name.startswith(("_", ".")))

for d in ("output/sz_csv", "output/sz_parquet", "output/sz_zstd"):
    print(f"{d:<20} {dir_size(d):>6} bytes")
# (41 rows won't show Parquet's 5-10x win — column stats & headers dominate
#  at toy scale. The RATIO flips hard with real data; the mechanism is the point.)

## Malformed records: PERMISSIVE / DROPMALFORMED / FAILFAST

`events.jsonl` line 10 is corrupt on purpose.

In [ ]:
EVENTS = f"{DATA_DIR}/events.jsonl"

# PERMISSIVE + _corrupt_record: keep bad rows, capture their raw text
events_schema = "event_id BIGINT, ts STRING, action STRING, _corrupt_record STRING"
permissive = spark.read.schema(events_schema).json(EVENTS)
permissive.filter(col("_corrupt_record").isNotNull()).show(truncate=60)

In [ ]:
# DROPMALFORMED: silently loses the row — count the damage yourself
dropped = spark.read.option("mode", "DROPMALFORMED").json(EVENTS)
print("rows kept:", dropped.count(), "of 12")

# FAILFAST: loud, immediate — the production contract mode
try:
    spark.read.option("mode", "FAILFAST").json(EVENTS).count()
except Exception as e:
    print("FAILFAST ->", str(e).splitlines()[0][:100])

## CSV dialect survival kit

In [ ]:
# A hostile European CSV: semicolons, comma decimals-as-strings, NA nulls, quoted quotes
hostile = 'id;name;price;note\n1;Kaffee;"18,50";NA\n2;Tee;"6,75";"mit ""Zitrone"""\n'
pathlib.Path("output").mkdir(exist_ok=True)
pathlib.Path("output/hostile.csv").write_text(hostile, encoding="utf-8")

(spark.read
 .option("header", True)
 .option("sep", ";")
 .option("quote", '"')
 .option("escape", '"')
 .option("nullValue", "NA")
 .csv("output/hostile.csv")
).show(truncate=False)

## Cleanup

In [ ]:
import shutil
shutil.rmtree("output", ignore_errors=True)
print("cleaned up.")

## Exercises

1. Round-trip `orders` through JSON (`write.json` / `read.json`). What happened to `order_date`'s type, and why does Parquet not have this problem?
2. Write `orders` as Parquet with `maxRecordsPerFile=10` — how many files now? Where might you use this for real?
3. Re-read the parquet output giving an explicit schema that OMITS three columns. Does it error? What does this tell you about column pruning (preview of 5.2)?
4. Rerun the same `mode("append")` parquet write twice and count the result — the duplicate problem that makes `overwrite`+idempotent design a Capstone 1 theme.

In [ ]:
# your exercise code here
